In [22]:
import pandas as pd

In [ ]:
tpu_data = pd.read_csv("Data/Used Data/techpowerup_gpus_with_images.csv")
price_data = pd.read_csv("Data/Used Data/all_scraped_gpu_prices.csv")
ub_data = pd.read_csv("Data/Used Data/GPU_UserBenchmarks.csv")

In [24]:
def format_ub_name(row):
    brand = str(row['Brand'])
    model = str(row['Model'])
    
    # Fix UserBenchmark's specific hyphen and bracket formats
    model = model.replace('-S (Super)', 'SUPER')
    model = model.replace('-TS (Ti-Super)', 'Ti SUPER')
    model = model.replace('-', ' ') # Converts -Ti into Ti, -XTX into XTX
    
    if brand == 'Nvidia': return 'GeForce ' + model
    elif brand == 'AMD': return 'Radeon ' + model
    elif brand == 'Intel': return 'Arc ' + model
    return model

In [25]:
ub_data['Mapped_Name'] = ub_data.apply(format_ub_name, axis=1)

ub_avg = ub_data.groupby('Mapped_Name')[['Benchmark']].mean().reset_index()

ub_mapped_names = sorted(ub_avg['Mapped_Name'].unique(), key=len, reverse=True)

In [26]:
def find_best_ub_match(tpu_name):
    if tpu_name in ub_mapped_names:
        return tpu_name
        
    for ub_name in ub_mapped_names:
        if tpu_name.startswith(ub_name + " "):
            return ub_name
            
    return None

In [27]:
tpu_data['UB_Match_Name'] = tpu_data['Name'].apply(find_best_ub_match)

category_prices = price_data[['GPU_Category', 'Average_Category_Price_IDR']].drop_duplicates()

dataset = pd.merge(
    tpu_data, 
    category_prices, 
    left_on='Name', 
    right_on='GPU_Category', 
    how='inner'
)

# FIXED: Changed left_on from 'Name' to 'UB_Match_Name'
dataset = pd.merge(
    dataset, 
    ub_avg, 
    left_on='UB_Match_Name', 
    right_on='Mapped_Name', 
    how='left'
)

In [28]:
dataset['Benchmark'] = dataset['Benchmark'].fillna(0)

dataset = dataset.drop(columns=['GPU_Category', 'Mapped_Name', 'UB_Match_Name']).rename(
    columns={'Average_Category_Price_IDR': 'Average Price (IDR)'}
)


dataset.to_csv("Data/dataset.csv", index=False)

print(f"Total GPUs matched and saved: {len(dataset)}")
display(dataset.head())

Total GPUs matched and saved: 45


,Name,Image_URL,Release Date,Bus,Memory,GPU Clock,Memory Clock,Cores / TMUs / ROPs,Average Price (IDR),Benchmark
0,Radeon RX 9070 XT,https://tpucdn.com/gpu-specs/images-new/c/4229...,"Mar 6th, 2025",PCIe 5.0 x16,16 GB / GDDR6 / 256 bit,2970 MHz,2518 MHz,4096 / 256 / 128,14302368,97.3
1,GeForce RTX 5090,https://tpucdn.com/gpu-specs/images-new/c/4216...,"Jan 30th, 2025",PCIe 5.0 x16,32 GB / GDDR7 / 512 bit,2407 MHz,1750 MHz,21760 / 680 / 176,61334900,183.0
2,GeForce RTX 5070,https://tpucdn.com/gpu-specs/images-new/c/4218...,"Mar 4th, 2025",PCIe 5.0 x16,12 GB / GDDR7 / 192 bit,2512 MHz,1750 MHz,6144 / 192 / 80,12956452,98.5
3,Radeon RX 9060 XT 16 GB,https://tpucdn.com/gpu-specs/images-new/c/4293...,"Jun 4th, 2025",PCIe 5.0 x16,16 GB / GDDR6 / 128 bit,3130 MHz,2518 MHz,2048 / 128 / 64,8822500,64.2
4,GeForce RTX 5070 Ti,https://tpucdn.com/gpu-specs/images-new/c/4243...,"Feb 20th, 2025",PCIe 5.0 x16,16 GB / GDDR7 / 256 bit,2452 MHz,1750 MHz,8960 / 280 / 96,17224308,119.0
